# Task 7 — Reddito, istruzione e stato di salute

**Domanda di ricerca:** Esiste una relazione tra il livello di reddito, l'istruzione e lo stato di salute delle persone?

**Fonte dei dati:** Eurostat, indagine EU-SILC. Due tabelle (entrambe in percentuale di persone):

| File | Tabella Eurostat | Cosa misura | Dimensione socio-economica |
|---|---|---|---|
| T7.1 | `hlth_silc_11` | Persone con una malattia o un problema di salute di lunga durata | Quintile di reddito |
| T7.2 | `hlth_silc_02` | Salute percepita (molto cattiva … molto buona) | Livello di istruzione (ISCED 2011) |

Le due tabelle condividono **paese, anno, sesso e classe di età**: sono queste le chiavi che useremo per metterle in relazione.

**Fasi della pipeline:**
1. Librerie e percorsi dei file
2. Lettura dei file CSV
3. Esplorazione iniziale
4. Pulizia (colonne vuote, colonne costanti, etichette ridondanti, traduzione in italiano, valori mancanti, tipi di dato)
5. Controlli di validità
6. Preparazione dei dati per rispondere alla domanda
7. Salvataggio dei file puliti
8. Verifica preliminare

In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [43]:
# Gestione dinamica e flessibile dei percorsi per evitare errori di FileNotFoundError
BASE_DIR = Path(".").resolve()

# Ricerca automatica dei file sia nella cartella attuale che in sottocartelle comuni
possibili_percorsi_reddito = [
    BASE_DIR / "T7.1_hlth_silc_11__custom_23966893_linear_2_0.csv",
    BASE_DIR / "data" / "T7.1_hlth_silc_11__custom_23966893_linear_2_0.csv",
    BASE_DIR / "notebook" / "data" / "T7.1_hlth_silc_11__custom_23966893_linear_2_0.csv",
    BASE_DIR.parent / "notebook" / "data" / "T7.1_hlth_silc_11__custom_23966893_linear_2_0.csv",
]

possibili_percorsi_istruzione = [
    BASE_DIR / "T7.2_hlth_silc_02__custom_23967814_linear_2_0.csv",
    BASE_DIR / "data" / "T7.2_hlth_silc_02__custom_23967814_linear_2_0.csv",
    BASE_DIR / "notebook" / "data" / "T7.2_hlth_silc_02__custom_23967814_linear_2_0.csv",
    BASE_DIR.parent / "notebook" / "data" / "T7.2_hlth_silc_02__custom_23967814_linear_2_0.csv",
]

PERCORSO_REDDITO = None
for p in possibili_percorsi_reddito:
    if p.exists():
        PERCORSO_REDDITO = p
        break

PERCORSO_ISTRUZIONE = None
for p in possibili_percorsi_istruzione:
    if p.exists():
        PERCORSO_ISTRUZIONE = p
        break

if PERCORSO_REDDITO is None or PERCORSO_ISTRUZIONE is None:
    raise FileNotFoundError(f"File CSV non trovati nella cartella attuale: {BASE_DIR}")

# Cartella in cui verranno salvati i file puliti
CARTELLA_OUTPUT = PERCORSO_REDDITO.parent / "puliti"

# Impostazione per l'esclusione o la segnalazione della bassa affidabilità (flag "u")
ESCLUDI_BASSA_AFFIDABILITA = False

print(f"Percorso T7.1 verificato: {PERCORSO_REDDITO.resolve()}")
print(f"Percorso T7.2 verificato: {PERCORSO_ISTRUZIONE.resolve()}")

Percorso T7.1 verificato: C:\Users\user\Desktop\Project_Work_DAT2\Analisi-e-previsione-delle-tendenze-della-salute\notebook\data\T7.1_hlth_silc_11__custom_23966893_linear_2_0.csv
Percorso T7.2 verificato: C:\Users\user\Desktop\Project_Work_DAT2\Analisi-e-previsione-delle-tendenze-della-salute\notebook\data\T7.2_hlth_silc_02__custom_23967814_linear_2_0.csv


In [44]:
df_reddito_grezzo = pd.read_csv(PERCORSO_REDDITO, delimiter=",")
df_istruzione_grezzo = pd.read_csv(PERCORSO_ISTRUZIONE, delimiter=",")

print(f"T7.1 (reddito e malattie di lunga durata): {df_reddito_grezzo.shape[0]} righe x {df_reddito_grezzo.shape[1]} colonne")
print(f"T7.2 (istruzione e salute percepita):      {df_istruzione_grezzo.shape[0]} righe x {df_istruzione_grezzo.shape[1]} colonne")

T7.1 (reddito e malattie di lunga durata): 25120 righe x 23 colonne
T7.2 (istruzione e salute percepita):      75120 righe x 25 colonne


In [45]:
def riepilogo_qualita(df):
    """Tabella riassuntiva: tipo, valori nulli e valori distinti di ogni colonna."""
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "valori_nulli": df.isna().sum(),
        "pct_nulli": (df.isna().mean() * 100).round(1),
        "valori_distinti": df.nunique(dropna=True),
    })

print("=== T7.1 (reddito) ===")
display(riepilogo_qualita(df_reddito_grezzo))
print("=== T7.2 (istruzione) ===")
display(riepilogo_qualita(df_istruzione_grezzo))

=== T7.1 (reddito) ===


,tipo,valori_nulli,pct_nulli,valori_distinti
STRUCTURE,str,0,0.0,1
STRUCTURE_ID,str,0,0.0,1
STRUCTURE_NAME,str,0,0.0,1
freq,str,0,0.0,1
Time frequency,str,0,0.0,1
unit,str,0,0.0,1
Unit of measure,str,0,0.0,1
quant_inc,str,0,0.0,5
Income quantile,str,0,0.0,5
age,str,0,0.0,8


=== T7.2 (istruzione) ===


,tipo,valori_nulli,pct_nulli,valori_distinti
STRUCTURE,str,0,0.0,1
STRUCTURE_ID,str,0,0.0,1
STRUCTURE_NAME,str,0,0.0,1
freq,str,0,0.0,1
Time frequency,str,0,0.0,1
unit,str,0,0.0,1
Unit of measure,str,0,0.0,1
isced11,str,0,0.0,3
International Standard Classification of Education (ISCED 2011),str,0,0.0,3
age,str,0,0.0,8


In [46]:
df_reddito = df_reddito_grezzo.copy()
df_istruzione = df_istruzione_grezzo.copy()

In [47]:
def rimuovi_colonne_vuote(df, nome):
    """Elimina le colonne in cui tutti i valori sono nulli."""
    vuote = df.columns[df.isna().all()].tolist()
    print(f"[{nome}] colonne completamente vuote rimosse ({len(vuote)}): {vuote}")
    return df.drop(columns=vuote)

df_reddito = rimuovi_colonne_vuote(df_reddito, "reddito")
df_istruzione = rimuovi_colonne_vuote(df_istruzione, "istruzione")

[reddito] colonne completamente vuote rimosse (4): ['Time', 'Observation value', 'CONF_STATUS', 'Confidentiality status (flag)']
[istruzione] colonne completamente vuote rimosse (4): ['Time', 'Observation value', 'CONF_STATUS', 'Confidentiality status (flag)']


In [48]:
COLONNE_CHIAVE = ["geo", "TIME_PERIOD", "sex", "age"]

def rimuovi_colonne_costanti(df, nome, da_mantenere=COLONNE_CHIAVE):
    """Elimina le colonne con un solo valore distinto, stampando il valore eliminato."""
    costanti = [c for c in df.columns
                if df[c].nunique(dropna=True) == 1 and c not in da_mantenere]
    print(f"[{nome}] colonne costanti rimosse ({len(costanti)}):")
    for c in costanti:
        print(f"    {c} = {df[c].dropna().iloc[0]!r}")
    return df.drop(columns=costanti)

df_reddito = rimuovi_colonne_costanti(df_reddito, "reddito")
df_istruzione = rimuovi_colonne_costanti(df_istruzione, "istruzione")

[reddito] colonne costanti rimosse (7):
    STRUCTURE = 'dataflow'
    STRUCTURE_ID = 'ESTAT:HLTH_SILC_11(1.0)'
    STRUCTURE_NAME = 'People having a long-standing illness or health problem, by sex, age and income quintile'
    freq = 'A'
    Time frequency = 'Annual'
    unit = 'PC'
    Unit of measure = 'Percentage'
[istruzione] colonne costanti rimosse (7):
    STRUCTURE = 'dataflow'
    STRUCTURE_ID = 'ESTAT:HLTH_SILC_02(1.0)'
    STRUCTURE_NAME = 'Self-perceived health by sex, age and educational attainment level'
    freq = 'A'
    Time frequency = 'Annual'
    unit = 'PC'
    Unit of measure = 'Percentage'


In [49]:
def rimuovi_spazi(df):
    """Toglie gli spazi iniziali e finali da tutte le colonne di testo."""
    for colonna in df.columns:
        if pd.api.types.is_string_dtype(df[colonna]):
            df[colonna] = df[colonna].str.strip()
    return df

def verifica_e_rimuovi_etichette(df, coppie_codice_etichetta, nome):
    """Controlla che ogni codice abbia una sola etichetta, poi elimina le etichette."""
    for codice_col, etichetta_col in coppie_codice_etichetta.items():
        etichette_per_codice = df.groupby(codice_col)[etichetta_col].nunique().max()
        if etichette_per_codice != 1:
            raise ValueError(f"[{nome}] il codice '{codice_col}' ha più di un'etichetta in '{etichetta_col}'")
    etichette = list(coppie_codice_etichetta.values())
    print(f"[{nome}] corrispondenza codice-etichetta verificata; rimosse {len(etichette)} etichette inglesi")
    return df.drop(columns=etichette)

ETICHETTE_REDDITO = {
    "quant_inc": "Income quantile",
    "age": "Age class",
    "sex": "Sex",
    "geo": "Geopolitical entity (reporting)",
    "OBS_FLAG": "Observation status (Flag) V2 structure",
}
ETICHETTE_ISTRUZIONE = {
    "isced11": "International Standard Classification of Education (ISCED 2011)",
    "levels": "Level",
    "age": "Age class",
    "sex": "Sex",
    "geo": "Geopolitical entity (reporting)",
    "OBS_FLAG": "Observation status (Flag) V2 structure",
}

df_reddito = rimuovi_spazi(df_reddito)
df_istruzione = rimuovi_spazi(df_istruzione)
df_reddito = verifica_e_rimuovi_etichette(df_reddito, ETICHETTE_REDDITO, "reddito")
df_istruzione = verifica_e_rimuovi_etichette(df_istruzione, ETICHETTE_ISTRUZIONE, "istruzione")

[reddito] corrispondenza codice-etichetta verificata; rimosse 5 etichette inglesi
[istruzione] corrispondenza codice-etichetta verificata; rimosse 6 etichette inglesi


In [50]:
DIZ_PAESI = {
    "AT": "Austria", "BE": "Belgio", "BG": "Bulgaria", "CY": "Cipro", "CZ": "Cechia",
    "DE": "Germania", "DK": "Danimarca", "EE": "Estonia", "EL": "Grecia", "ES": "Spagna",
    "FI": "Finlandia", "FR": "Francia", "HR": "Croazia", "HU": "Ungheria", "IE": "Irlanda",
    "IS": "Islanda", "IT": "Italia", "LT": "Lituania", "LU": "Lussemburgo", "LV": "Lettonia",
    "MT": "Malta", "NL": "Paesi Bassi", "NO": "Norvegia", "PL": "Polonia", "PT": "Portogallo",
    "RO": "Romania", "SE": "Svezia", "SI": "Slovenia", "SK": "Slovacchia",
}

DIZ_SESSO = {"F": "Femmine", "M": "Maschi"}

DIZ_ETA = {
    "Y16-24": "16-24 anni", "Y25-34": "25-34 anni", "Y35-44": "35-44 anni", "Y45-54": "45-54 anni",
    "Y55-64": "55-64 anni", "Y65-74": "65-74 anni", "Y75-84": "75-84 anni", "Y_GE85": "85 anni e oltre",
}

DIZ_QUINTILI = {
    "QU1": "1° quintile (reddito più basso)", "QU2": "2° quintile", "QU3": "3° quintile",
    "QU4": "4° quintile", "QU5": "5° quintile (reddito più alto)",
}
DIZ_NUMERO_QUINTILE = {"QU1": 1, "QU2": 2, "QU3": 3, "QU4": 4, "QU5": 5}

DIZ_ISTRUZIONE = {
    "ED0-2": "Bassa (ISCED 0-2)",
    "ED3_4": "Media (ISCED 3-4)",
    "ED5-8": "Alta (ISCED 5-8)",
}

DIZ_SALUTE = {
    "VBAD": "Molto cattiva", "BAD": "Cattiva", "FAIR": "Discreta",
    "GOOD": "Buona", "VGOOD": "Molto buona",
}

DIZ_FLAG = {
    "b": "interruzione nella serie storica",
    "bu": "interruzione nella serie storica, bassa affidabilità",
    "e": "stimato",
    "m": "valore mancante, il dato non può esistere",
    "n": "non significativo",
    "p": "provvisorio",
    "pu": "provvisorio, bassa affidabilità",
    "u": "bassa affidabilità",
}

ORDINE_ETA = list(DIZ_ETA.values())
ORDINE_QUINTILI = list(DIZ_QUINTILI.values())
ORDINE_ISTRUZIONE = list(DIZ_ISTRUZIONE.values())
ORDINE_SALUTE = list(DIZ_SALUTE.values())

def traduci_colonna(df, colonna, dizionario):
    """Traduce i codici di una colonna; se trova un codice sconosciuto si ferma con un errore."""
    tradotta = df[colonna].map(dizionario)
    sconosciuti = df.loc[tradotta.isna() & df[colonna].notna(), colonna].unique()
    if len(sconosciuti) > 0:
        raise ValueError(f"Codici non presenti nel dizionario per '{colonna}': {list(sconosciuti)}")
    return tradotta

# --- Traduzioni Reddito (T7.1)
df_reddito = df_reddito.rename(columns={
    "quant_inc": "quintile", "age": "classe_eta", "sex": "sesso", "geo": "codice_paese",
    "TIME_PERIOD": "anno", "OBS_VALUE": "pct_malattia_lunga_durata", "OBS_FLAG": "flag",
})
df_reddito["paese"] = traduci_colonna(df_reddito, "codice_paese", DIZ_PAESI)
df_reddito["numero_quintile"] = traduci_colonna(df_reddito, "quintile", DIZ_NUMERO_QUINTILE)
df_reddito["quintile"] = traduci_colonna(df_reddito, "quintile", DIZ_QUINTILI)
df_reddito["classe_eta"] = traduci_colonna(df_reddito, "classe_eta", DIZ_ETA)
df_reddito["sesso"] = traduci_colonna(df_reddito, "sesso", DIZ_SESSO)

# --- Traduzioni Istruzione (T7.2)
df_istruzione = df_istruzione.rename(columns={
    "isced11": "livello_istruzione", "levels": "livello_salute", "age": "classe_eta", "sex": "sesso",
    "geo": "codice_paese", "TIME_PERIOD": "anno", "OBS_VALUE": "pct_salute_percepita", "OBS_FLAG": "flag",
})
df_istruzione["paese"] = traduci_colonna(df_istruzione, "codice_paese", DIZ_PAESI)
df_istruzione["livello_istruzione"] = traduci_colonna(df_istruzione, "livello_istruzione", DIZ_ISTRUZIONE)
df_istruzione["livello_salute"] = traduci_colonna(df_istruzione, "livello_salute", DIZ_SALUTE)
df_istruzione["classe_eta"] = traduci_colonna(df_istruzione, "classe_eta", DIZ_ETA)
df_istruzione["sesso"] = traduci_colonna(df_istruzione, "sesso", DIZ_SESSO)

df_reddito = df_reddito[["codice_paese", "paese", "anno", "sesso", "classe_eta",
                         "numero_quintile", "quintile", "pct_malattia_lunga_durata", "flag"]]
df_istruzione = df_istruzione[["codice_paese", "paese", "anno", "sesso", "classe_eta",
                               "livello_istruzione", "livello_salute", "pct_salute_percepita", "flag"]]

df_reddito.head()

,codice_paese,paese,anno,sesso,classe_eta,numero_quintile,quintile,pct_malattia_lunga_durata,flag
0,AT,Austria,2015,Femmine,16-24 anni,1,1° quintile (reddito più basso),16.4,NaN
1,AT,Austria,2016,Femmine,16-24 anni,1,1° quintile (reddito più basso),32.0,NaN
2,AT,Austria,2017,Femmine,16-24 anni,1,1° quintile (reddito più basso),36.7,NaN
3,AT,Austria,2018,Femmine,16-24 anni,1,1° quintile (reddito più basso),36.2,NaN
4,AT,Austria,2019,Femmine,16-24 anni,1,1° quintile (reddito più basso),40.6,NaN


In [51]:
def elimina_righe_senza_valore(df, colonna_valore, nome):
    """Elimina le righe con valore nullo, mostrando prima i flag che le accompagnano."""
    senza_valore = df[colonna_valore].isna()
    print(f"[{nome}] righe senza valore: {senza_valore.sum()} su {len(df)} ({senza_valore.mean() * 100:.1f}%)")
    motivi = (df.loc[senza_valore, "flag"].fillna("(nessun flag)")
                .map(lambda f: DIZ_FLAG.get(f, f)).value_counts())
    print(motivi.to_string())
    print()
    return df.loc[~senza_valore].reset_index(drop=True)

df_reddito = elimina_righe_senza_valore(df_reddito, "pct_malattia_lunga_durata", "reddito")
df_istruzione = elimina_righe_senza_valore(df_istruzione, "pct_salute_percepita", "istruzione")

[reddito] righe senza valore: 4212 su 25120 (16.8%)
flag
bassa affidabilità                                      4162
valore mancante, il dato non può esistere                 48
interruzione nella serie storica, bassa affidabilità       2

[istruzione] righe senza valore: 2060 su 75120 (2.7%)
flag
bassa affidabilità                           2040
valore mancante, il dato non può esistere      20



In [52]:
for df in (df_reddito, df_istruzione):
    df["bassa_affidabilita"] = df["flag"].fillna("").str.contains("u")

print(f"Reddito:    {df_reddito['bassa_affidabilita'].mean() * 100:.1f}% delle osservazioni valide ha bassa affidabilità")
print(f"Istruzione: {df_istruzione['bassa_affidabilita'].mean() * 100:.1f}% delle osservazioni valide ha bassa affidabilità")

print("\nQuota di bassa affidabilità per classe d'età (reddito):")
display((df_reddito.groupby("classe_eta")["bassa_affidabilita"].mean() * 100).round(1))

if ESCLUDI_BASSA_AFFIDABILITA:
    df_reddito = df_reddito.loc[~df_reddito["bassa_affidabilita"]].reset_index(drop=True)
    df_istruzione = df_istruzione.loc[~df_istruzione["bassa_affidabilita"]].reset_index(drop=True)
    print("\nOsservazioni a bassa affidabilità ELIMINATE.")
else:
    print("\nOsservazioni a bassa affidabilità MANTENUTE (solo segnalate).")

Reddito:    31.9% delle osservazioni valide ha bassa affidabilità
Istruzione: 6.8% delle osservazioni valide ha bassa affidabilità

Quota di bassa affidabilità per classe d'età (reddito):


classe_eta
16-24 anni          3.1
25-34 anni         68.8
35-44 anni         64.9
45-54 anni         37.4
55-64 anni         13.5
65-74 anni         16.1
75-84 anni         29.4
85 anni e oltre    55.9
Name: bassa_affidabilita, dtype: float64


Osservazioni a bassa affidabilità MANTENUTE (solo segnalate).


In [53]:
def imposta_tipi(df, colonne_ordinate):
    """Imposta l'anno come intero e le colonne ordinali come categorie ordinate."""
    df["anno"] = df["anno"].astype(int)
    for colonna, ordine in colonne_ordinate.items():
        df[colonna] = pd.Categorical(df[colonna], categories=ordine, ordered=True)
    return df

df_reddito = imposta_tipi(df_reddito, {"classe_eta": ORDINE_ETA, "quintile": ORDINE_QUINTILI})
df_istruzione = imposta_tipi(df_istruzione, {"classe_eta": ORDINE_ETA,
                                             "livello_istruzione": ORDINE_ISTRUZIONE,
                                             "livello_salute": ORDINE_SALUTE})

print(df_reddito.dtypes)
print()
print(df_istruzione.dtypes)

codice_paese                      str
paese                             str
anno                            int64
sesso                             str
classe_eta                   category
numero_quintile                 int64
quintile                     category
pct_malattia_lunga_durata     float64
flag                              str
bassa_affidabilita               bool
dtype: object

codice_paese                 str
paese                        str
anno                       int64
sesso                        str
classe_eta              category
livello_istruzione      category
livello_salute          category
pct_salute_percepita     float64
flag                         str
bassa_affidabilita          bool
dtype: object


In [54]:
CHIAVI_COMUNI = ["codice_paese", "anno", "sesso", "classe_eta"]

# 1) Le percentuali devono stare tra 0 e 100
for df, colonna, nome in [(df_reddito, "pct_malattia_lunga_durata", "reddito"),
                          (df_istruzione, "pct_salute_percepita", "istruzione")]:
    fuori_intervallo = ((df[colonna] < 0) | (df[colonna] > 100)).sum()
    assert fuori_intervallo == 0, f"[{nome}] {fuori_intervallo} percentuali fuori dall'intervallo 0-100"
    print(f"[{nome}] percentuali comprese tra 0 e 100: OK")

# 2) Nessun duplicato sulle chiavi univoche
duplicati_reddito = df_reddito.duplicated(subset=CHIAVI_COMUNI + ["numero_quintile"]).sum()
duplicati_istruzione = df_istruzione.duplicated(subset=CHIAVI_COMUNI + ["livello_istruzione", "livello_salute"]).sum()
assert duplicati_reddito == 0 and duplicati_istruzione == 0, "Trovate righe duplicate"
print("Nessuna riga duplicata: OK")

# 3) Anni presenti
print("Anni presenti (reddito):   ", sorted(df_reddito["anno"].unique().tolist()))
print("Anni presenti (istruzione):", sorted(df_istruzione["anno"].unique().tolist()))

# 4) Controllo somma percentuali della salute percepita
somme = (df_istruzione.groupby(CHIAVI_COMUNI + ["livello_istruzione"], observed=True)["pct_salute_percepita"]
         .agg(["sum", "count"]))
gruppi_completi = somme[somme["count"] == 5]
fuori_tolleranza = (np.abs(gruppi_completi["sum"] - 100) > 0.5).sum()
print(f"Gruppi con tutte e 5 le categorie di salute: {len(gruppi_completi)}; "
      f"con somma lontana da 100 (> 0,5): {fuori_tolleranza}")

[reddito] percentuali comprese tra 0 e 100: OK
[istruzione] percentuali comprese tra 0 e 100: OK
Nessuna riga duplicata: OK
Anni presenti (reddito):    [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Anni presenti (istruzione): [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Gruppi con tutte e 5 le categorie di salute: 14612; con somma lontana da 100 (> 0,5): 0


In [55]:
def tabella_copertura(grezzo, chiave_grezzo, pulito, chiave_pulito):
    """Righe originali, righe valide e percentuale di righe valide per ogni valore della chiave."""
    totali = grezzo.groupby(chiave_grezzo).size()
    valide = pulito.groupby(chiave_pulito, observed=True).size().reindex(totali.index).fillna(0).astype(int)
    return pd.DataFrame({"righe_totali": totali, "righe_valide": valide,
                         "pct_valide": (valide / totali * 100).round(1)})

print("=== Copertura per anno (reddito) ===")
display(tabella_copertura(df_reddito_grezzo, "TIME_PERIOD", df_reddito, "anno"))

print("=== Paesi con la copertura più bassa (reddito) ===")
display(tabella_copertura(df_reddito_grezzo, "geo", df_reddito, "codice_paese").sort_values("pct_valide").head(8))

print("=== Paesi con la copertura più bassa (istruzione) ===")
display(tabella_copertura(df_istruzione_grezzo, "geo", df_istruzione, "codice_paese").sort_values("pct_valide").head(8))

=== Copertura per anno (reddito) ===


,righe_totali,righe_valide,pct_valide
TIME_PERIOD,,,
2015,2320,2108,90.9
2016,2320,1904,82.1
2017,2320,1908,82.2
2018,2320,1878,80.9
2019,2320,1892,81.6
2020,2320,1791,77.2
2021,2240,1813,80.9
2022,2240,1888,84.3
2023,2240,1888,84.3


=== Paesi con la copertura più bassa (reddito) ===


,righe_totali,righe_valide,pct_valide
geo,,,
IS,480,189,39.4
LU,880,600,68.2
DK,880,604,68.6
RO,880,632,71.8
MT,880,641,72.8
BG,880,657,74.7
LT,880,663,75.3
IE,880,676,76.8


=== Paesi con la copertura più bassa (istruzione) ===


,righe_totali,righe_valide,pct_valide
geo,,,
IS,1440,1195,83.0
LU,2640,2400,90.9
MT,2640,2440,92.4
FI,2640,2480,93.9
CY,2640,2500,94.7
PT,2640,2505,94.9
NO,2640,2520,95.5
LT,2640,2530,95.8


In [56]:
reddito_largo = df_reddito.pivot_table(index=CHIAVI_COMUNI, columns="numero_quintile",
                                       values="pct_malattia_lunga_durata", aggfunc="first", observed=True)
reddito_largo.columns = [f"malattia_q{q}" for q in reddito_largo.columns]
reddito_largo["divario_malattia_q1_q5"] = reddito_largo["malattia_q1"] - reddito_largo["malattia_q5"]
reddito_largo = reddito_largo.reset_index()

reddito_largo.head()

,codice_paese,anno,sesso,classe_eta,malattia_q1,malattia_q2,malattia_q3,malattia_q4,malattia_q5,divario_malattia_q1_q5
0,AT,2015,Femmine,16-24 anni,16.4,12.3,10.9,18.2,11.6,4.8
1,AT,2015,Femmine,25-34 anni,18.1,18.1,18.1,15.4,10.3,7.8
2,AT,2015,Femmine,35-44 anni,38.3,27.7,28.6,29.2,17.2,21.1
3,AT,2015,Femmine,45-54 anni,49.3,45.8,42.5,30.8,29.0,20.3
4,AT,2015,Femmine,55-64 anni,60.1,45.6,46.9,42.6,35.0,25.1


In [57]:
salute_cattiva = (df_istruzione[df_istruzione["livello_salute"].isin(["Molto cattiva", "Cattiva"])]
                  .groupby(CHIAVI_COMUNI + ["livello_istruzione"], observed=True)["pct_salute_percepita"]
                  .agg(["sum", "count"]).reset_index())
salute_cattiva = salute_cattiva[salute_cattiva["count"] == 2]

NOMI_COLONNE_ISTRUZIONE = {
    "Bassa (ISCED 0-2)": "salute_cattiva_istr_bassa",
    "Media (ISCED 3-4)": "salute_cattiva_istr_media",
    "Alta (ISCED 5-8)": "salute_cattiva_istr_alta",
}

istruzione_largo = salute_cattiva.pivot_table(index=CHIAVI_COMUNI, columns="livello_istruzione",
                                              values="sum", aggfunc="first", observed=True)
istruzione_largo.columns = [NOMI_COLONNE_ISTRUZIONE[c] for c in istruzione_largo.columns]
istruzione_largo["divario_salute_cattiva_bassa_alta"] = (istruzione_largo["salute_cattiva_istr_bassa"]
                                                         - istruzione_largo["salute_cattiva_istr_alta"])
istruzione_largo = istruzione_largo.reset_index()

istruzione_largo.head()

,codice_paese,anno,sesso,classe_eta,salute_cattiva_istr_bassa,salute_cattiva_istr_media,salute_cattiva_istr_alta,divario_salute_cattiva_bassa_alta
0,AT,2015,Femmine,16-24 anni,1.1,0.2,0.6,0.5
1,AT,2015,Femmine,25-34 anni,7.9,1.9,0.8,7.1
2,AT,2015,Femmine,35-44 anni,13.9,5.7,2.3,11.6
3,AT,2015,Femmine,45-54 anni,22.9,6.0,2.6,20.3
4,AT,2015,Femmine,55-64 anni,21.6,7.9,6.0,15.6


In [58]:
dataset_integrato = reddito_largo.merge(istruzione_largo, on=CHIAVI_COMUNI, how="inner")
dataset_integrato.insert(1, "paese", dataset_integrato["codice_paese"].map(DIZ_PAESI))

print(f"Gruppi nel dataset integrato: {len(dataset_integrato)}")
print(f"Gruppi con entrambi i divari calcolabili: "
      f"{dataset_integrato[['divario_malattia_q1_q5', 'divario_salute_cattiva_bassa_alta']].dropna().shape[0]}")
dataset_integrato.head()

Gruppi nel dataset integrato: 4645
Gruppi con entrambi i divari calcolabili: 3618


,codice_paese,paese,anno,sesso,classe_eta,malattia_q1,malattia_q2,malattia_q3,malattia_q4,malattia_q5,divario_malattia_q1_q5,salute_cattiva_istr_bassa,salute_cattiva_istr_media,salute_cattiva_istr_alta,divario_salute_cattiva_bassa_alta
0,AT,Austria,2015,Femmine,16-24 anni,16.4,12.3,10.9,18.2,11.6,4.8,1.1,0.2,0.6,0.5
1,AT,Austria,2015,Femmine,25-34 anni,18.1,18.1,18.1,15.4,10.3,7.8,7.9,1.9,0.8,7.1
2,AT,Austria,2015,Femmine,35-44 anni,38.3,27.7,28.6,29.2,17.2,21.1,13.9,5.7,2.3,11.6
3,AT,Austria,2015,Femmine,45-54 anni,49.3,45.8,42.5,30.8,29.0,20.3,22.9,6.0,2.6,20.3
4,AT,Austria,2015,Femmine,55-64 anni,60.1,45.6,46.9,42.6,35.0,25.1,21.6,7.9,6.0,15.6


In [59]:
CARTELLA_OUTPUT.mkdir(parents=True, exist_ok=True)

df_reddito.to_csv(CARTELLA_OUTPUT / "reddito_pulito.csv", index=False, encoding="utf-8")
df_istruzione.to_csv(CARTELLA_OUTPUT / "istruzione_pulita.csv", index=False, encoding="utf-8")
dataset_integrato.to_csv(CARTELLA_OUTPUT / "dataset_integrato.csv", index=False, encoding="utf-8")

print(f"File salvati in: {CARTELLA_OUTPUT.resolve()}")
for nome in ("reddito_pulito.csv", "istruzione_pulita.csv", "dataset_integrato.csv"):
    print("  -", nome)

File salvati in: C:\Users\user\Desktop\Project_Work_DAT2\Analisi-e-previsione-delle-tendenze-della-salute\notebook\data\puliti
  - reddito_pulito.csv
  - istruzione_pulita.csv
  - dataset_integrato.csv


In [60]:
colonne_q = [f"malattia_q{q}" for q in range(1, 6)]
colonne_istr = ["salute_cattiva_istr_bassa", "salute_cattiva_istr_media", "salute_cattiva_istr_alta"]

print("% media di persone con malattia di lunga durata, per quintile di reddito")
print("(solo gruppi con tutti e 5 i quintili disponibili)")
display(reddito_largo[colonne_q].dropna().mean().round(1).rename("pct_media"))

print("% media di persone con salute cattiva o molto cattiva, per livello di istruzione")
print("(solo gruppi con tutti e 3 i livelli disponibili)")
display(istruzione_largo[colonne_istr].dropna().mean().round(1).rename("pct_media"))

print("Correlazione (Spearman) tra divario di reddito e divario di istruzione, sui gruppi integrati:")
display(dataset_integrato[["divario_malattia_q1_q5", "divario_salute_cattiva_bassa_alta"]]
        .corr(method="spearman").round(2))

% media di persone con malattia di lunga durata, per quintile di reddito
(solo gruppi con tutti e 5 i quintili disponibili)


malattia_q1    48.1
malattia_q2    44.3
malattia_q3    40.9
malattia_q4    38.1
malattia_q5    34.6
Name: pct_media, dtype: float64

% media di persone con salute cattiva o molto cattiva, per livello di istruzione
(solo gruppi con tutti e 3 i livelli disponibili)


salute_cattiva_istr_bassa    17.2
salute_cattiva_istr_media    10.8
salute_cattiva_istr_alta      7.0
Name: pct_media, dtype: float64

Correlazione (Spearman) tra divario di reddito e divario di istruzione, sui gruppi integrati:


,divario_malattia_q1_q5,divario_salute_cattiva_bassa_alta
divario_malattia_q1_q5,1.00,0.26
divario_salute_cattiva_bassa_alta,0.26,1.00


In [61]:
# 1. Tabella Formato Largo - Salute per Quintile di Reddito (Q1 - Q5)
print("=== TABELLA DIVARIO REDDITO (Q1 - Q5) ===")
display(reddito_largo)

# 2. Tabella Formato Largo - Salute Cattiva per Livello di Istruzione (Bassa - Alta)
print("\n=== TABELLA DIVARIO ISTRUZIONE (Bassa - Alta) ===")
display(istruzione_largo)

# 3. Tabella Dataset Integrato (Pronto per l'ispezione su Data Wrangler)
print("\n=== DATASET INTEGRATO COMPLETO ===")
display(dataset_integrato)

=== TABELLA DIVARIO REDDITO (Q1 - Q5) ===


,codice_paese,anno,sesso,classe_eta,malattia_q1,malattia_q2,malattia_q3,malattia_q4,malattia_q5,divario_malattia_q1_q5
0,AT,2015,Femmine,16-24 anni,16.4,12.3,10.9,18.2,11.6,4.8
1,AT,2015,Femmine,25-34 anni,18.1,18.1,18.1,15.4,10.3,7.8
2,AT,2015,Femmine,35-44 anni,38.3,27.7,28.6,29.2,17.2,21.1
3,AT,2015,Femmine,45-54 anni,49.3,45.8,42.5,30.8,29.0,20.3
4,AT,2015,Femmine,55-64 anni,60.1,45.6,46.9,42.6,35.0,25.1
...,...,...,...,...,...,...,...,...,...,...
4656,SK,2025,Maschi,35-44 anni,27.4,21.0,14.7,10.8,12.7,14.7
4657,SK,2025,Maschi,45-54 anni,36.9,38.7,24.5,23.5,17.3,19.6
4658,SK,2025,Maschi,55-64 anni,65.1,48.7,45.4,48.8,29.8,35.3
4659,SK,2025,Maschi,65-74 anni,79.7,72.4,69.8,63.8,43.0,36.7



=== TABELLA DIVARIO ISTRUZIONE (Bassa - Alta) ===


,codice_paese,anno,sesso,classe_eta,salute_cattiva_istr_bassa,salute_cattiva_istr_media,salute_cattiva_istr_alta,divario_salute_cattiva_bassa_alta
0,AT,2015,Femmine,16-24 anni,1.1,0.2,0.6,0.5
1,AT,2015,Femmine,25-34 anni,7.9,1.9,0.8,7.1
2,AT,2015,Femmine,35-44 anni,13.9,5.7,2.3,11.6
3,AT,2015,Femmine,45-54 anni,22.9,6.0,2.6,20.3
4,AT,2015,Femmine,55-64 anni,21.6,7.9,6.0,15.6
...,...,...,...,...,...,...,...,...
4993,SK,2025,Maschi,45-54 anni,35.4,3.7,1.0,34.4
4994,SK,2025,Maschi,55-64 anni,42.0,12.7,8.2,33.8
4995,SK,2025,Maschi,65-74 anni,44.1,20.7,14.7,29.4
4996,SK,2025,Maschi,75-84 anni,70.5,42.6,23.1,47.4



=== DATASET INTEGRATO COMPLETO ===


,codice_paese,paese,anno,sesso,classe_eta,malattia_q1,malattia_q2,malattia_q3,malattia_q4,malattia_q5,divario_malattia_q1_q5,salute_cattiva_istr_bassa,salute_cattiva_istr_media,salute_cattiva_istr_alta,divario_salute_cattiva_bassa_alta
0,AT,Austria,2015,Femmine,16-24 anni,16.4,12.3,10.9,18.2,11.6,4.8,1.1,0.2,0.6,0.5
1,AT,Austria,2015,Femmine,25-34 anni,18.1,18.1,18.1,15.4,10.3,7.8,7.9,1.9,0.8,7.1
2,AT,Austria,2015,Femmine,35-44 anni,38.3,27.7,28.6,29.2,17.2,21.1,13.9,5.7,2.3,11.6
3,AT,Austria,2015,Femmine,45-54 anni,49.3,45.8,42.5,30.8,29.0,20.3,22.9,6.0,2.6,20.3
4,AT,Austria,2015,Femmine,55-64 anni,60.1,45.6,46.9,42.6,35.0,25.1,21.6,7.9,6.0,15.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4640,SK,Slovacchia,2025,Maschi,35-44 anni,27.4,21.0,14.7,10.8,12.7,14.7,16.6,2.7,1.5,15.1
4641,SK,Slovacchia,2025,Maschi,45-54 anni,36.9,38.7,24.5,23.5,17.3,19.6,35.4,3.7,1.0,34.4
4642,SK,Slovacchia,2025,Maschi,55-64 anni,65.1,48.7,45.4,48.8,29.8,35.3,42.0,12.7,8.2,33.8
4643,SK,Slovacchia,2025,Maschi,65-74 anni,79.7,72.4,69.8,63.8,43.0,36.7,44.1,20.7,14.7,29.4
